# imports

NOTE: parts of the code used in this notebook were adapted from prior work in the context of the research project 'Digitaler Hass' at HTW Berlin

In [1]:
import sys
# adjust path
sys.path.append('../../../../NER-german-telegram')
import re
import pandas as pd
from src.helpers.db_helpers import execute_sql_select
import src.config.db_credentials as db

from datetime import datetime
from random import sample

from langdetect import detect, detect_langs
import hashlib

from langdetect import DetectorFactory
DetectorFactory.seed = 0

In [2]:
import emoji

# get data from Telegram channel 'Demotermine'

In [3]:
table_name = "telegram_seeds"
channel_name = 'Demotermine'
ts_before = datetime.now()
query = f"""SELECT * FROM {table_name} WHERE channel_name = '{channel_name}'"""
data = execute_sql_select(command=query, database=db.DB_NAME_TELEGRAM, return_result_as_df=True)
ts_after = datetime.now()

print(f"Took {ts_after - ts_before}")

Column names:  ['channel_name', 'channel_id', 'channel_description', 'message_id', 'from_id', 'via_bot_id', 'date', 'edit_date', 'text', 'forwards', 'fwd_from', 'replies', 'reply_to', 'media', 'views', 'id']
Connection to DB closed
Took 0:00:09.893799


In [4]:
df = data.copy()

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31430 entries, 0 to 31429
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   channel_name         31430 non-null  object        
 1   channel_id           31430 non-null  object        
 2   channel_description  31430 non-null  object        
 3   message_id           31430 non-null  object        
 4   from_id              0 non-null      object        
 5   via_bot_id           6 non-null      object        
 6   date                 31430 non-null  datetime64[ns]
 7   edit_date            29450 non-null  datetime64[ns]
 8   text                 28650 non-null  object        
 9   forwards             31429 non-null  object        
 10  fwd_from             1776 non-null   object        
 11  replies              25438 non-null  object        
 12  reply_to             23 non-null     object        
 13  media                30774 non-

# Data cleaning

## Drop NaN values and text duplicates

In [6]:
df = df.dropna(subset=['text'])

In [7]:
len(df)

28650

In [8]:
df = df.drop_duplicates(subset=['text']) # keeps first by default

In [9]:
len(df)

27627

## Anonymization: Replace IBANS, USER mentions

In [10]:
def replace_mentions_urls_ibans(x): # or import from bertopic_helpers: replace_mentions_urls_ibans
    x = re.sub(r'@\S+', 'USER', x)
    x = re.sub(r'(http|ftp|https):\/\/([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:\/~+#-]*[\w@?^=%&\/~+#-])', 'URL', x)
    x = re.sub(r'[tT]\.[Mm][eE]\/[-a-zA-Z0-9.]+(\/\S*)?', 'URL', x) # t.me
    x = re.sub(r'(www|WWW).[-a-zA-Z0-9.]+(\/\S*)?', 'URL', x) # www.
    x = re.sub(r'[A-Z]{2}[0-9]{2}(?:\s?[0-9]{4}){4}(?:\s?[0-9]{1,2})?', 'SENSITIVE', x) # iban
#   x = " ".join(x.split()) # Delete repeated whitespaces 
    return x

def remove_user_mentions_ibans(x):
    x = re.sub(r'@\S+', '', x)
    x = re.sub(r'[A-Z]{2}[0-9]{2}(?:\s?[0-9]{4}){4}(?:\s?[0-9]{1,2})?', '', x) # iban
#   x = " ".join(x.split()) # Delete repeated whitespaces 
    return x

# alternativ:
def remove_mentions_urls_ibans(x):
    x = re.sub(r'@\S+', '', x)
    x = re.sub(r'(http|ftp|https):\/\/([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:\/~+#-]*[\w@?^=%&\/~+#-])', '', x)
    x = re.sub(r'[tT]\.[Mm][eE]\/[-a-zA-Z0-9.]+(\/\S*)?', '', x) # t.me
    x = re.sub(r'(www|WWW).[-a-zA-Z0-9.]+(\/\S*)?', '', x) # www.
    x = re.sub(r'[A-Z]{2}[0-9]{2}(?:\s?[0-9]{4}){4}(?:\s?[0-9]{1,2})?', '', x) # iban
#   x = " ".join(x.split()) # Delete repeated whitespaces 
    return x

In [11]:
df['text'] = df['text'].apply(remove_mentions_urls_ibans)

df_no_dups = df.drop_duplicates(subset=['text'])

In [12]:
len(df)

27627

## Normalize unicode strings, remove emojis, keep umlauts

In [13]:
from unicodedata import normalize
import spacy
nlp = spacy.load("de_core_news_md")

In [14]:
def is_ascii(s):
    """Check if the characters in string s are in ASCII, U+0-U+7F."""
    return len(s) == len(s.encode())

In [15]:
def normalize_string_keep_umlauts(string_lowercase, umlauts = ['ä', 'ö', 'ü', 'ß']):
    """Normalize unicode strings, e.g. with special formatting, but keep umlauts"""
    out = []
    
    for c in string_lowercase: 
        if c in umlauts: 
            pass
        else: 
            c = normalize('NFKD', c)
            
        out.append(c)       
    return "".join([c for c in out])

In [16]:
def preprocess_text(s):

    s = ' '.join(s.split())
    
    # only normalize strings which are not already in ascii-format
    if not is_ascii(s):
        s = normalize_string_keep_umlauts(s)
        s = emoji.replace_emoji(s, replace='')
    
    doc = nlp(s)
   
    words = [token.text for token in doc] #if token.is_alpha or token.is_digit or token.is_punct]
        
    return " ".join(words)

In [17]:
# give_emoji_free_text(df.text.loc[1])

In [18]:
df['cleaned_text'] = None

In [19]:
df['cleaned_text'] = df['text'].apply(lambda x: preprocess_text(x) if len(x)> 0 else None)

In [20]:
df = df.dropna(subset=['cleaned_text'])

In [97]:
len(df)

27613

In [21]:
df.to_csv('../temp/demotermine_cleaned_2022_06_09.csv')

In [22]:
df = pd.read_csv('../temp/demotermine_cleaned_2022_06_09.csv', parse_dates=['date', 'edit_date'])

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27586 entries, 0 to 27585
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Unnamed: 0           27586 non-null  int64         
 1   channel_name         27586 non-null  object        
 2   channel_id           27586 non-null  int64         
 3   channel_description  27586 non-null  object        
 4   message_id           27586 non-null  int64         
 5   from_id              0 non-null      float64       
 6   via_bot_id           5 non-null      float64       
 7   date                 27586 non-null  datetime64[ns]
 8   edit_date            26530 non-null  datetime64[ns]
 9   text                 27586 non-null  object        
 10  forwards             27586 non-null  float64       
 11  fwd_from             1439 non-null   object        
 12  replies              24592 non-null  object        
 13  reply_to             19 non-nul

## Text length

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27586 entries, 0 to 27585
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Unnamed: 0.1         27586 non-null  int64         
 1   Unnamed: 0           27586 non-null  int64         
 2   channel_name         27586 non-null  object        
 3   channel_id           27586 non-null  int64         
 4   channel_description  27586 non-null  object        
 5   message_id           27586 non-null  int64         
 6   from_id              0 non-null      float64       
 7   via_bot_id           5 non-null      float64       
 8   date                 27586 non-null  datetime64[ns]
 9   edit_date            26530 non-null  datetime64[ns]
 10  text                 27586 non-null  object        
 11  forwards             27586 non-null  float64       
 12  fwd_from             1439 non-null   object        
 13  replies              24592 non-

In [40]:
df.dropna(subset=['cleaned_text'], inplace=True)

In [41]:
df['text_length'] = None

In [42]:
df['text_length'] = df.cleaned_text.apply(lambda x: len(x))

In [43]:
df.to_csv('../temp/demotermine_cleaned_2022_06_09.csv')

In [44]:
df = pd.read_csv('../temp/demotermine_cleaned_2022_06_09.csv', parse_dates=['date', 'edit_date'])

# Language detection

In [45]:
def detect_language(s):    
    try: 
        return detect(s)
        
    except:
        return None

In [46]:
df['lang'] = df.cleaned_text.apply(lambda x: detect_language(x))

In [47]:
df.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,channel_name,channel_id,channel_description,message_id,from_id,via_bot_id,date,...,forwards,fwd_from,replies,reply_to,media,views,id,cleaned_text,lang,text_length
0,0,0,0,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,15971,NaN,NaN,2021-08-29 11:11:25,...,0.0,NaN,"MessageReplies(replies=0, replies_pts=290294, ...",NaN,NaN,1779.0,Demotermine1250288610159712021-08-29 11:10:12,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,de,213
1,1,1,2,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,23371,NaN,NaN,2021-10-26 09:58:45,...,1.0,NaN,"MessageReplies(replies=0, replies_pts=290292, ...",NaN,MessageMediaDocument(document=Document(id=6323...,1834.0,Demotermine1250288610233712021-10-26 09:58:28,Vic Aus Dan Andrews caught bluffing again .....,en,162
2,2,2,3,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,10449,NaN,NaN,2021-05-13 06:52:14,...,2.0,NaN,"MessageReplies(replies=0, replies_pts=290300, ...",NaN,MessageMediaPhoto(photo=Photo(id=5239960968883...,2226.0,Demotermine1250288610104492021-05-13 07:02:17,"World Wide Demonstration 2.0 Ukraine , Ki...",en,380
3,3,3,4,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,6885,NaN,NaN,2021-02-27 07:11:14,...,1.0,NaN,"MessageReplies(replies=0, replies_pts=290300, ...",NaN,MessageMediaWebPage(webpage=WebPage(id=2944466...,4373.0,Demotermine125028861068852021-02-27 07:11:32,27 Feb : Sternmarsch – Graz BEHÖRDLICH VERB...,de,152
4,4,4,5,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,22413,NaN,NaN,2021-10-20 04:49:20,...,7.0,NaN,"MessageReplies(replies=0, replies_pts=290292, ...",NaN,MessageMediaPhoto(photo=Photo(id=5438483832002...,1203.0,Demotermine1250288610224132021-10-20 04:48:43,Raus auf die Straßen Übersicht / Overview,de,44


In [48]:
languages = df.groupby('lang').size().sort_values(ascending=False)

In [49]:
languages.to_csv('../temp/language_count_2022_06_09-2.csv')

In [50]:
df_de = df[df['lang'] == 'de']

In [51]:
len(df_de)

23929

In [52]:
df_de.reset_index(inplace=True, drop=True)

In [53]:
#df_de.drop(columns=['Unnamed: 0.2','Unnamed: 0.1', 'Unnamed: 0'], inplace=True)

In [54]:
df_de.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,channel_name,channel_id,channel_description,message_id,from_id,via_bot_id,date,...,forwards,fwd_from,replies,reply_to,media,views,id,cleaned_text,lang,text_length
0,0,0,0,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,15971,NaN,NaN,2021-08-29 11:11:25,...,0.0,NaN,"MessageReplies(replies=0, replies_pts=290294, ...",NaN,NaN,1779.0,Demotermine1250288610159712021-08-29 11:10:12,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,de,213
1,3,3,4,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,6885,NaN,NaN,2021-02-27 07:11:14,...,1.0,NaN,"MessageReplies(replies=0, replies_pts=290300, ...",NaN,MessageMediaWebPage(webpage=WebPage(id=2944466...,4373.0,Demotermine125028861068852021-02-27 07:11:32,27 Feb : Sternmarsch – Graz BEHÖRDLICH VERB...,de,152
2,4,4,5,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,22413,NaN,NaN,2021-10-20 04:49:20,...,7.0,NaN,"MessageReplies(replies=0, replies_pts=290292, ...",NaN,MessageMediaPhoto(photo=Photo(id=5438483832002...,1203.0,Demotermine1250288610224132021-10-20 04:48:43,Raus auf die Straßen Übersicht / Overview,de,44
3,5,5,6,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,4836,NaN,NaN,2021-01-01 03:50:02,...,9.0,NaN,"MessageReplies(replies=0, replies_pts=290300, ...",NaN,MessageMediaWebPage(webpage=WebPage(id=1503448...,4999.0,Demotermine125028861048362021-01-01 03:57:43,Stuttgart-Stream von Stephan Bergmann u.a. mit...,de,101
4,6,6,7,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,3105,NaN,NaN,2020-10-09 13:28:26,...,2.0,NaN,"MessageReplies(replies=0, replies_pts=290300, ...",NaN,MessageMediaWebPage(webpage=WebPage(id=1823867...,4108.0,Demotermine125028861031052020-10-09 13:28:27,Demo-Termine von Zum Redaktionsschluss um 00:0...,de,786


In [55]:
df_de.to_csv('../temp/demotermine_cleaned_2022_06_09_DE.csv')

In [56]:
df_de.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,channel_name,channel_id,channel_description,message_id,from_id,via_bot_id,date,...,forwards,fwd_from,replies,reply_to,media,views,id,cleaned_text,lang,text_length
0,0,0,0,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,15971,NaN,NaN,2021-08-29 11:11:25,...,0.0,NaN,"MessageReplies(replies=0, replies_pts=290294, ...",NaN,NaN,1779.0,Demotermine1250288610159712021-08-29 11:10:12,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,de,213
1,3,3,4,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,6885,NaN,NaN,2021-02-27 07:11:14,...,1.0,NaN,"MessageReplies(replies=0, replies_pts=290300, ...",NaN,MessageMediaWebPage(webpage=WebPage(id=2944466...,4373.0,Demotermine125028861068852021-02-27 07:11:32,27 Feb : Sternmarsch – Graz BEHÖRDLICH VERB...,de,152
2,4,4,5,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,22413,NaN,NaN,2021-10-20 04:49:20,...,7.0,NaN,"MessageReplies(replies=0, replies_pts=290292, ...",NaN,MessageMediaPhoto(photo=Photo(id=5438483832002...,1203.0,Demotermine1250288610224132021-10-20 04:48:43,Raus auf die Straßen Übersicht / Overview,de,44
3,5,5,6,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,4836,NaN,NaN,2021-01-01 03:50:02,...,9.0,NaN,"MessageReplies(replies=0, replies_pts=290300, ...",NaN,MessageMediaWebPage(webpage=WebPage(id=1503448...,4999.0,Demotermine125028861048362021-01-01 03:57:43,Stuttgart-Stream von Stephan Bergmann u.a. mit...,de,101
4,6,6,7,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,3105,NaN,NaN,2020-10-09 13:28:26,...,2.0,NaN,"MessageReplies(replies=0, replies_pts=290300, ...",NaN,MessageMediaWebPage(webpage=WebPage(id=1823867...,4108.0,Demotermine125028861031052020-10-09 13:28:27,Demo-Termine von Zum Redaktionsschluss um 00:0...,de,786
